# 실험 05: LangChain 챗봇 메모리의 종류와 구현

## 1. 실험 제목과 목표
- **제목**: Short-term Memory와 Long-term Memory의 결합
- **목표**: 챗봇 메모리를 '현재 대화 맥락(단기)'과 '사용자 고유 정보(장기)'로 나누어 이해합니다. LangChain의 `RunnableWithMessageHistory`로 단기 기억을 자동화하고, 템플릿 변수를 통해 장기 기억을 주입하는 실무적인 패턴을 구현해봅니다.

## 2. 메모리의 종류 (개념 정리)
실제 서비스에서 챗봇의 메모리는 다음과 같이 세분화됩니다.

| 구분 | 설명 | 예시 |
|---|---|---|
| **Short-term Memory** | 현재 대화 안의 맥락 유지 | "방금 설명한 RAG를 표로 바꿔줘" |
| **Long-term Memory** | 사용자 선호나 프로젝트 정보 저장 (DB 연동) | "나는 코딩할 때 무조건 Python만 써." |
| **State / Checkpointer**| 현재 작업 상태 및 실패 후 재개 (LangGraph) | 여러 Tool을 거치는 과정에서의 중간 상태 저장 |

*이 노트북에서는 `Short-term`과 `Long-term` 메모리를 결합하는 방법을 다룹니다. (State와 Checkpointer는 추후 LangGraph 챕터에서 다룹니다.)*

## 3. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

## 4. 환경 변수 로드 및 모델 준비

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

## 5. 비교 실험 1: 단기 기억(Short-term Memory) 전용 챗봇
이전 질문의 맥락을 이어가는 가장 기본적인 챗봇입니다.

In [3]:
# 1. 프롬프트에 'history' 빈 공간 뚫어주기
short_term_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 다정한 도우미입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])

chain_short = short_term_prompt | llm | parser

# 2. 세션 관리를 위한 저장소 (인메모리)
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 3. 자동화 래퍼 씌우기
chatbot_short = RunnableWithMessageHistory(
    chain_short,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

config = {"configurable": {"session_id": "User_001"}}

print("[대화 1]")
print("AI:", chatbot_short.invoke({"question": "서울의 대표적인 산 3개만 말해줘."}, config=config))

print("\n[대화 2 - 지시대명사 사용]")
print("AI:", chatbot_short.invoke({"question": "방금 말한 것들을 영어로 번역해줘."}, config=config))
print("\n-> 결과: '방금 말한 것'이 무엇인지 Short-term Memory를 통해 정확히 인지합니다.")

c:\Users\USER\Desktop\llm-study-lab\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


[대화 1]
AI: 서울의 대표적인 산으로는 다음의 세 가지를 들 수 있습니다:

1. **남산**: 서울의 중심에 위치해 있으며, N서울타워가 있어 관광 명소로 유명합니다. 남산 공원도 잘 조성되어 있어 많은 사람들이 방문합니다.

2. **북한산**: 서울과 경기도의 경계에 위치한 북한산은 다양한 등산로와 아름다운 자연 경관이 특징입니다. 또한, 여러 사찰과 문화유산이 있어 많은 등산객들이 찾습니다.

3. **관악산**: 서울대와 가까워 많은 학생들이 찾는 산입니다. 관악산 정상에서는 서울 시내의 전경을 한눈에 볼 수 있어, 등산객들에게 인기 있는 장소입니다.

이 세 산은 각각 고유한 매력을 가지고 있어 많은 사람들이 사랑하는 곳입니다!

[대화 2 - 지시대명사 사용]
AI: Sure! Here are the translations of the three representative mountains in Seoul:

1. **Namsan**: Located in the center of Seoul, Namsan is famous for the N Seoul Tower and is a popular tourist destination. Namsan Park is also well-developed, attracting many visitors.

2. **Bukhansan**: Situated on the boundary between Seoul and Gyeonggi Province, Bukhansan is known for its various hiking trails and beautiful natural scenery. It also hosts several temples and cultural heritage sites, making it a popular spot for hikers.

3. **Gwanaksan**: Close to Seoul National University, Gwanaksan is frequently visited b

## 6. 비교 실험 2: 장기 기억(Long-term Memory) 결합
챗봇이 사용자의 고유한 정보(이름, 선호도, 직업 등)를 영구적으로 기억하는 것처럼 작동하게 만듭니다. 실제로는 DB(예: RDBMS나 Vector DB)에서 유저 프로필을 꺼내와 System Prompt에 주입(Injection)하는 방식입니다.

In [4]:
# 가짜 데이터베이스 (실제로는 MySQL, MongoDB, Pinecone 등에서 사용자 프로필을 불러옵니다)
fake_user_db = {
    "User_001": "이름은 김개발. 직업은 파이썬 백엔드 개발자. 모든 코드는 객체지향적으로 작성하는 것을 선호함.",
    "User_002": "이름은 박디자인. 직업은 UI/UX 디자이너. 코드보다는 시각적인 비유를 좋아함."
}

# 프롬프트에 Long-term (user_profile)과 Short-term (history)을 모두 뚫어줍니다.
long_term_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 맞춤형 AI 도우미입니다.\n사용자 정보: {user_profile}"),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{question}")
])

chain_long = long_term_prompt | llm | parser

chatbot_combined = RunnableWithMessageHistory(
    chain_long,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

def chat_with_memory(session_id, user_question):
    # 1. DB에서 사용자 장기 기억(프로필)을 불러옴
    user_profile = fake_user_db.get(session_id, "알 수 없는 사용자")
    
    # 2. 챗봇 호출 시 단기 기억(Session ID) 설정과 장기 기억(user_profile) 주입을 동시에 진행
    return chatbot_combined.invoke(
        {"question": user_question, "user_profile": user_profile},
        config={"configurable": {"session_id": session_id}}
    )

print("=== [User_001 (김개발) 접속] ===")
print("AI:", chat_with_memory("User_001", "웹 서버를 만들려면 어떻게 해야 해? 짧게 핵심만."))

print("\n=== [User_002 (박디자인) 접속] ===")
print("AI:", chat_with_memory("User_002", "웹 서버를 만들려면 어떻게 해야 해? 짧게 핵심만."))

print("\n-> 결과: 똑같은 질문을 했지만, System Prompt에 주입된 Long-term Memory(사용자 직업/선호도)에 따라 완전히 다른 맞춤형 답변이 나옵니다!")

c:\Users\USER\Desktop\llm-study-lab\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


=== [User_001 (김개발) 접속] ===
AI: 웹 서버를 만들기 위해서는 다음의 핵심 단계를 따라야 합니다:

1. **프로그래밍 언어 선택**: Python, Node.js, Java 등 웹 서버를 구축할 언어를 선택합니다.

2. **웹 프레임워크 사용**: Flask, Django (Python), Express (Node.js)와 같은 웹 프레임워크를 이용하여 개발을 시작합니다.

3. **라우팅 설정**: URL과 해당 요청에 대한 처리를 정의하는 라우팅을 설정합니다.

4. **서버 실행**: `app.run()` (Flask)이나 `server.listen()` (Node.js)을 사용하여 서버를 실행합니다.

5. **테스트**: Postman이나 웹 브라우저를 사용해 API나 웹 페이지를 테스트합니다.

이렇게 하면 기본적인 웹 서버를 구축할 수 있습니다!

=== [User_002 (박디자인) 접속] ===
AI: 웹 서버를 만들기 위해서는 다음과 같은 핵심 단계를 따르면 됩니다:

1. **언어 선택**: Python, Node.js, Ruby 등 서버 사이드 언어 중 하나를 선택합니다.
2. **프레임워크 사용**: Express.js (Node.js), Django (Python) 같은 프레임워크를 사용해 기본 설정과 라우팅을 관리합니다.
3. **서버 환경 준비**: 로컬 개발 환경 또는 클라우드 서비스(AWS, Heroku 등)에 서버 환경을 설정합니다.
4. **코드 작성**: 요청을 처리하고 응답을 반환하는 코드 작성.
5. **테스트 및 배포**: 로컬에서 테스트 후, 클라우드 또는 호스팅 서비스에 배포합니다.

이 과정을 통해 간단한 웹 서버를 구축할 수 있습니다!

-> 결과: 똑같은 질문을 했지만, System Prompt에 주입된 Long-term Memory(사용자 직업/선호도)에 따라 완전히 다른 맞춤형 답변이 나옵니다!


## 7. 실패/주의 케이스: 세션 간의 기억 고립 (Isolation)
각 세션(Session ID)의 단기 기억은 철저히 격리됩니다. 이를 실수로 섞거나 잘못 호출하면 기억이 유실된 것처럼 보입니다.

In [5]:
print("[김개발(User_001)이 두 번째 질문을 함]")
# 정상적인 경우: 방금 한 질문(웹 서버)의 맥락을 이어감
print("정상 AI:", chat_with_memory("User_001", "방금 말한 방법 중에서 가장 쉬운 프레임워크는 뭐야?"))

print("\n[만약 개발자가 session_id 관리를 실수해서 User_003(비회원)으로 넘겼다면?]")
print("오류 AI:", chat_with_memory("User_003", "방금 말한 방법 중에서 가장 쉬운 프레임워크는 뭐야?"))
print("\n-> 결과: 세션이 다르면 이전 대화 내역(history)이 비어있기 때문에 '방금 말한 방법'이 무엇인지 전혀 알지 못합니다.")

[김개발(User_001)이 두 번째 질문을 함]
정상 AI: 가장 쉬운 웹 프레임워크는 **Flask**입니다. Flask는 Python으로 작성된 마이크로 프레임워크로, 간단한 애플리케이션을 빠르게 만들 수 있게 도와줍니다. 

Flask의 장점은 다음과 같습니다:
- **간결함**: 코드가 직관적이고 적은 양의 코드로 시작할 수 있습니다.
- **유연성**: 필요한 기능만 추가할 수 있어, 기본 사용이 간편합니다.
- **풍부한 문서화**: 공식 문서와 커뮤니티 자료가 잘 마련되어 있어 배우기 쉽습니다.

Flask로 쉽게 웹 서버를 구축하고 실습해보는 것을 추천드립니다!

[만약 개발자가 session_id 관리를 실수해서 User_003(비회원)으로 넘겼다면?]
오류 AI: 가장 쉬운 프레임워크는 사용자의 경험과 필요에 따라 다를 수 있지만, 일반적으로 다음과 같은 프레임워크들이 배우기 쉬운 것으로 알려져 있습니다:

1. **Flask**: 파이썬 기반의 마이크로 프레임워크로, 간단하고 직관적인 구조 덕분에 빠르게 시작할 수 있습니다. 필요에 따라 확장하기도 쉽습니다.

2. **Ruby on Rails**: 루비 언어로 작성된 웹 애플리케이션 프레임워크로, 강력한 규약과 설정보다 관습을 따르기 때문에 빠르게 개발할 수 있습니다.

3. **Django**: 역시 파이썬 기반으로, "배터리가 포함된" 프레임워크로 알려져 있으며, 기본적으로 많은 기능이 제공되어 개발 속도를 높여줍니다.

4. **Express.js**: Node.js 기반의 웹 프레임워크로, 심플하고 유연하여 빠른 프로토타입 제작에 적합합니다.

어떤 프레임워크가 가장 쉬운지는 주로 개인의 작업 환경과 선호도에 따라 다르므로, 몇 가지를 시도해보고 본인에게 맞는 것을 선택하는 것이 좋습니다.

-> 결과: 세션이 다르면 이전 대화 내역(history)이 비어있기 때문에 '방금 말한 방법'이 무엇인지 전혀 알지 못합니다.


## 8. 결과 해석

1. **Short-term Memory의 한계**: `RunnableWithMessageHistory`나 `ConversationBufferMemory`는 현재 대화방의 맥락을 유지하지만, 창을 끄거나 너무 오랜 시간이 지나면 사라지는(또는 토큰 한계로 지워지는) 휘발성 성격을 가집니다.
2. **Long-term Memory의 본질**: 진정한 개인화 챗봇(예: ChatGPT의 Custom Instructions)은 대화 기록 안에서 사용자의 특성을 추출해 DB(VectorDB 등)에 영구 저장한 뒤, 다음 대화 때 `System Prompt`로 찔러 넣어주는(Injection) 구조로 동작합니다.
3. **State / Checkpointer**: 복잡한 에이전트(Agent)가 문서를 읽고, 코드를 짜고, 검증하는 등 여러 단계를 거칠 때는 단순한 대화 기록이 아닌 '작업 상태'를 저장해야 하며 이는 LangGraph의 Checkpointer 기술로 해결합니다.

## 9. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 교재에서 말하는 메모리는 단순히 '이전 대화 기억하기(단기)'뿐만 아니라 '사용자 프로필 저장(장기)'과 '실행 상태 저장(State)'으로 나뉨을 이해함.
* `RunnableWithMessageHistory`로 단기 기억을 자동화하고, DB에서 가져온 사용자 정보를 템플릿 변수로 넘겨 장기 기억을 구현하는 아키텍처 패턴을 확인함.

### 다음 개선 방향
* 대화 내역이 너무 길어질 때 토큰 한계를 피하기 위해 과거 대화를 강제로 요약하는 `ConversationSummaryMemory` 계열 학습 필요.
* 나아가 사용자의 대화에서 선호도("나는 파이썬이 좋아")를 AI가 스스로 캐치해서 DB(Long-term)에 몰래 업데이트하는 Memory Agent 패턴 도입 고려.